In [ ]:
import requests

API_HOST = "domainr.p.rapidapi.com"
API_KEY = "d0c57c7d69mshd3740bdbe30a37dp1a35c8jsn81117c113ab0"

def check_domain_availability(domain):
    url = f"https://{API_HOST}/v2/status"
    headers = {
        "x-rapidapi-host": API_HOST,
        "x-rapidapi-key": API_KEY
    }
    params = {"domain": domain}

    resp = requests.get(url, headers=headers, params=params, timeout=10)
    resp.raise_for_status()
    return resp.json()

In [3]:
import string

tld = ".fo"  # Specify the TLD you want to check

# Generate all two-letter combinations (aa to zz)
base_names = [a + b for a in string.ascii_lowercase for b in string.ascii_lowercase]
domains = [f"{base}{tld}" for base in base_names]

print(f"Total domains generated: {len(domains)}")

Total domains generated: 676


In [4]:
import csv
import os

# Check which domains have already been processed
processed_domains = set()
csv_file = "data/domains.csv"

if os.path.exists(csv_file):
    with open(csv_file, "r", newline="") as f:
        reader = csv.reader(f)
        next(reader)  # Skip header
        for row in reader:
            if row:  # Make sure row is not empty
                processed_domains.add(row[0])  # domain is in first column

print(f"Already processed {len(processed_domains)} domains")

# Filter out already processed domains
remaining_domains = [d for d in domains if d not in processed_domains]
print(f"Remaining domains to process: {len(remaining_domains)}")

# Resume processing with append mode
with open(csv_file, "a", newline="") as f:
    writer = csv.writer(f)
    
    # Only write header if file is empty/new
    if len(processed_domains) == 0:
        writer.writerow(["domain", "zone", "status", "summary"])

    for i, domain in enumerate(remaining_domains):
        print(f"Processing {i+1}/{len(remaining_domains)}: {domain}")
        
        try:
            payload = check_domain_availability(domain)
            
            for entry in payload.get("status", []):
                writer.writerow([
                    entry.get("domain", ""),
                    entry.get("zone", ""),
                    entry.get("status", ""),
                    entry.get("summary", "")
                ])
            f.flush()
            
        except Exception as e:
            print(f"Error processing {domain}: {e}")
            continue

Already processed 0 domains
Remaining domains to process: 676
Processing 1/676: aa.fo
Processing 2/676: ab.fo
Processing 3/676: ac.fo
Processing 4/676: ad.fo
Processing 5/676: ae.fo
Processing 6/676: af.fo
Processing 7/676: ag.fo
Processing 8/676: ah.fo
Processing 9/676: ai.fo
Processing 10/676: aj.fo
Processing 11/676: ak.fo
Processing 12/676: al.fo
Processing 13/676: am.fo
Processing 14/676: an.fo
Processing 15/676: ao.fo
Processing 16/676: ap.fo
Processing 17/676: aq.fo
Processing 18/676: ar.fo
Processing 19/676: as.fo
Processing 20/676: at.fo
Processing 21/676: au.fo
Processing 22/676: av.fo
Processing 23/676: aw.fo
Processing 24/676: ax.fo
Processing 25/676: ay.fo
Processing 26/676: az.fo
Processing 27/676: ba.fo
Processing 28/676: bb.fo
Processing 29/676: bc.fo
Processing 30/676: bd.fo
Processing 31/676: be.fo
Processing 32/676: bf.fo
Processing 33/676: bg.fo
Processing 34/676: bh.fo
Processing 35/676: bi.fo
Processing 36/676: bj.fo
Processing 37/676: bk.fo
Processing 38/676: bl.

In [5]:
import csv, os

csv_file = "data/domains.csv"
rows = {}  # domain -> dict(zone,status,summary)

# Load existing rows (if any)
if os.path.exists(csv_file):
    with open(csv_file, "r", newline="") as f:
        r = csv.reader(f)
        header = next(r, None)
        for row in r:
            if not row: 
                continue
            dom = row[0].strip()
            rows[dom] = {
                "zone": row[1].strip() if len(row) > 1 else "",
                "status": row[2].strip().lower() if len(row) > 2 else "",
                "summary": row[3].strip() if len(row) > 3 else "",
            }

ERROR_STATUSES = {"", "error", "failed", "timeout", "unknown"}

# Figure out which to (re)process
to_fix = [d for d in domains if d not in rows or rows[d]["status"] in ERROR_STATUSES]
print(f"Retrying {len(to_fix)} domains (missing or previously errored)…")

# Re-check and update in-memory rows
for d in to_fix:
    try:
        payload = check_domain_availability(d) or {}
        entries = payload.get("status", [])
        # Use the first matching entry (fallbacks if API returns nothing)
        e = next((e for e in entries if e.get("domain") == d), entries[0] if entries else {})
        rows[d] = {
            "zone": e.get("zone", ""),
            "status": (e.get("status", "") or "").lower(),
            "summary": e.get("summary", ""),
        }
    except Exception as ex:
        rows[d] = {"zone": "", "status": "error", "summary": f"retry_failed: {ex}"}

# Write back a clean CSV (header + all domains)
os.makedirs(os.path.dirname(csv_file), exist_ok=True)
with open(csv_file, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["domain", "zone", "status", "summary"])
    for d in sorted(rows.keys()):
        r = rows[d]
        w.writerow([d, r["zone"], r["status"], r["summary"]])

print("Done.")


Retrying 0 domains (missing or previously errored)…
Done.


In [6]:
# Print any that still failed after retry
still_bad = [d for d, r in rows.items() if r["status"] in ERROR_STATUSES]
print(f"{len(still_bad)} domains still failed after retry.")

if still_bad:
    print("These domains still failed:", still_bad)
else:
    print("✅ All domains processed successfully")


0 domains still failed after retry.
✅ All domains processed successfully
